# 🚀 Think AI FULL SYSTEM with Docker in Colab

This uses Docker to run all databases reliably!

In [ ]:
# Step 1: Setup repository
import os

if os.path.exists("full_architecture_chat.py"):
    pass
elif os.path.exists("/content/think_ai"):
    %cd /content/think_ai
    !git pull
else:
    %cd /content
    !git clone https://github.com/champi-dev/Think_AI think_ai
    %cd think_ai

In [ ]:
# Step 2: Install Docker in Colab
%%bash
# Update packages
apt-get update -qq

# Install dependencies
apt-get install -y -qq apt-transport-https ca-certificates curl software-properties-common

# Add Docker GPG key
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | apt-key add -

# Add Docker repository
add-apt-repository "deb [arch=amd64] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable"

# Install Docker
apt-get update -qq
apt-get install -y -qq docker-ce

# Install Docker Compose
curl -L "https://github.com/docker/compose/releases/download/1.29.2/docker-compose-$(uname -s)-$(uname -m)" -o /usr/local/bin/docker-compose
chmod +x /usr/local/bin/docker-compose

echo "Docker installed!"

In [ ]:
# Step 3: Start Docker daemon
import subprocess
import time

# Start Docker daemon in background
docker_process = subprocess.Popen(["dockerd"],
                                 stdout=subprocess.DEVNULL,
                                 stderr=subprocess.DEVNULL)

time.sleep(10)

# Verify Docker is running
!docker version

In [ ]:
# Step 4: Create lightweight docker-compose for Colab
docker_compose_content = """version: "3.8"

services:
  redis:
    image: redis:7-alpine
    container_name: think-redis
    ports:
      - "6379:6379"
    command: redis-server --maxmemory 256mb --maxmemory-policy allkeys-lru

  cassandra:
    image: cassandra:4.1
    container_name: think-cassandra
    ports:
      - "9042:9042"
    environment:
      - CASSANDRA_CLUSTER_NAME=ThinkAI
      - MAX_HEAP_SIZE=512M
      - HEAP_NEWSIZE=128M

  neo4j:
    image: neo4j:5.16.0
    container_name: think-neo4j
    ports:
      - "7687:7687"
      - "7474:7474"
    environment:
      - NEO4J_AUTH=neo4j/thinkaipass
      - NEO4J_dbms_memory_heap_max__size=512M

  milvus:
    image: milvusdb/milvus:v2.3.5
    container_name: think-milvus
    command: milvus run standalone
    environment:
      ETCD_USE_EMBED: true
      COMMON_STORAGETYPE: local
    ports:
      - "19530:19530"
      - "9091:9091"
"""

with open("docker-compose-colab.yml", "w") as f:
    f.write(docker_compose_content)


In [ ]:
# Step 5: Start all services with Docker Compose
!docker-compose -f docker-compose-colab.yml up -d

# Wait for services to start
time.sleep(30)

# Check status
!docker-compose -f docker-compose-colab.yml ps

In [ ]:
# Step 6: Create .env file for full system
env_content = """# Google Colab FULL SYSTEM with Docker
ENVIRONMENT=colab_docker
LOG_LEVEL=INFO

# Real services via Docker - NO MOCKS!
USE_MOCK_SERVICES=false

# ScyllaDB/Cassandra
SCYLLA_HOSTS=localhost
SCYLLA_PORT=9042
SCYLLA_KEYSPACE=thinkaidb

# Redis
REDIS_HOST=localhost
REDIS_PORT=6379
REDIS_DB=0

# Milvus
MILVUS_HOST=localhost
MILVUS_PORT=19530

# Neo4j
NEO4J_URI=bolt://localhost:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=thinkaipass

# Model settings for GPU
MODEL_NAME=Qwen/Qwen2.5-Coder-1.5B-Instruct
DEVICE=cuda
MAX_TOKENS=250
"""

with open(".env", "w") as f:
    f.write(env_content)


In [ ]:
# Step 7: Install Python dependencies
!pip install -r requirements.txt

In [ ]:
# Step 8: Verify all services are running
!echo "🔍 Checking services..."
!docker exec think-redis redis-cli ping && echo "✅ Redis is running" || echo "❌ Redis not ready"
!docker exec think-cassandra nodetool status && echo "✅ Cassandra is running" || echo "❌ Cassandra not ready"
!docker exec think-neo4j cypher-shell -u neo4j -p thinkaipass "RETURN 1" && echo "✅ Neo4j is running" || echo "❌ Neo4j not ready"
!nc -zv localhost 19530 && echo "✅ Milvus is running" || echo "❌ Milvus not ready"

In [ ]:
# Step 9: RUN THE FULL SYSTEM!
!python full_architecture_chat.py

In [ ]:
# Optional: View logs if needed
# !docker-compose -f docker-compose-colab.yml logs

In [ ]:
# Cleanup when done
# !docker-compose -f docker-compose-colab.yml down